In [1]:
!pip uninstall -y torchao
!pip install -q torchao==0.16.0 transformers accelerate peft bitsandbytes pillow tqdm

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import gc

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model, PeftModel
from torch.optim import AdamW

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
BASE_DIR = "/content/drive/MyDrive/BanglaVision"

IMAGE_ROOT = f"{BASE_DIR}/bd_images"
JSON_PATH = f"{BASE_DIR}/bd_captions_bn_final.json"

BD_CHECKPOINT_DIR = f"{BASE_DIR}/bd_checkpoints"
os.makedirs(BD_CHECKPOINT_DIR, exist_ok=True)

In [5]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total BD samples:", len(data))

for item in data:
    if not item["image"].startswith("/"):
        item["image"] = os.path.join(IMAGE_ROOT, item["image"])

Total BD samples: 11593


In [6]:
class BDDataset(Dataset):
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        try:
            image = Image.open(item["image"]).convert("RGB")
        except:
            image = Image.new("RGB", (224, 224))

        text = "বাংলায় বিস্তারিত বর্ণনা কর: " + item["caption_bn"]

        encoding = self.processor(
            images=image,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=100
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = encoding["input_ids"].clone()

        return encoding

In [7]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

base_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)

base_model.config.use_cache = True

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(base_model, lora_config)

print("LoRA applied")
model.print_trainable_parameters()

LoRA applied
trainable params: 2,621,440 || all params: 3,747,383,296 || trainable%: 0.0700


In [9]:
def disable_all_checkpointing(model):
    for module in model.modules():
        if hasattr(module, "gradient_checkpointing"):
            module.gradient_checkpointing = False

disable_all_checkpointing(model)

In [11]:
def get_last_checkpoint():
    if not os.path.exists(BD_CHECKPOINT_DIR):
        return None

    checkpoints = []

    for ckpt in os.listdir(BD_CHECKPOINT_DIR):
        path = os.path.join(BD_CHECKPOINT_DIR, ckpt)

        if os.path.exists(os.path.join(path, "adapter_config.json")):
            checkpoints.append(ckpt)

    if len(checkpoints) == 0:
        return None

    checkpoints = sorted(checkpoints, key=lambda x: int(x.split("_")[-1]))
    return os.path.join(BD_CHECKPOINT_DIR, checkpoints[-1])


last_ckpt = get_last_checkpoint()

if last_ckpt:
    print("Resuming from:", last_ckpt)
    model = PeftModel.from_pretrained(model, last_ckpt)
else:
    print("Starting fresh training")

Starting fresh training


In [12]:
def split_data(data, step):
    return [data[i:i+step] for i in range(0, len(data), step)]

STEP_SIZE = 3000
splits = split_data(data, STEP_SIZE)

print("Total chunks:", len(splits))

Total chunks: 4


In [13]:
BATCH_SIZE = 1
EPOCHS = 3

for epoch in range(EPOCHS):

    print(f"\nBD EPOCH {epoch+1}")

    for i, split in enumerate(splits):

        checkpoint_path = os.path.join(
            BD_CHECKPOINT_DIR,
            f"bd_step_{epoch}_{(i+1)*STEP_SIZE}"
        )

        if os.path.exists(os.path.join(checkpoint_path, "adapter_model.safetensors")):
            print(f"Skipping chunk {i+1}")
            continue

        print(f"\nTraining BD chunk {i+1}/{len(splits)}")

        dataset = BDDataset(split, processor)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

        optimizer = AdamW(model.parameters(), lr=3e-5)

        model.train()

        pbar = tqdm(loader)

        for batch in pbar:
            batch = {k: v.to("cuda") for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            pbar.set_description(f"Loss: {loss.item():.4f}")

        model.save_pretrained(checkpoint_path)
        processor.save_pretrained(checkpoint_path)

        print(f"Saved: {checkpoint_path}")

        torch.cuda.empty_cache()
        gc.collect()


BD EPOCH 1

Training BD chunk 1/4


Loss: 0.0145: 100%|██████████| 3000/3000 [21:19<00:00,  2.35it/s]


Saved: /content/drive/MyDrive/BanglaVision/bd_checkpoints/bd_step_0_3000

Training BD chunk 2/4


Loss: 0.0186: 100%|██████████| 3000/3000 [37:05<00:00,  1.35it/s]


Saved: /content/drive/MyDrive/BanglaVision/bd_checkpoints/bd_step_0_6000

Training BD chunk 3/4


...

Saved: /content/drive/MyDrive/BanglaVision/bd_checkpoints/bd_step_2_6000

Training BD chunk 3/4


Loss: 0.0173: 100%|██████████| 3000/3000 [08:41<00:00,  5.76it/s]


Saved: /content/drive/MyDrive/BanglaVision/bd_checkpoints/bd_step_2_9000

Training BD chunk 4/4


Loss: 0.0807: 100%|██████████| 2593/2593 [07:30<00:00,  5.76it/s]


Saved: /content/drive/MyDrive/BanglaVision/bd_checkpoints/bd_step_2_12000
